In [1]:
import numpy as np
import matplotlib.pyplot as plt
import math
from dataclasses import dataclass
import pickle
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from torch.nn import functional as F

In [2]:
with open('tokenizers/wiki_token.pkl', 'rb') as f:
    wiki_data = pickle.load(f)

print(wiki_data[0])
print(len(wiki_data))

[354, 696, 76, 11404, 2821, 318, 257, 1316, 68, 13, 340, 318, 287, 443, 12, 2934, 12, 69, 8132, 287, 262, 3209, 47476, 5011, 287, 5093, 1216, 590, 13]
243883


In [111]:

# tokens and curriculum score come in, tensors of x and y go out and sample ids with difficulty
class DataLoader:
    def __init__(self, x, y, batch_size, max_len=100, pad_token=0):
        self.x, self.y = x, y
        self.batch_size = batch_size
        self.max_len = max_len
        self.pad_token = pad_token
        self.n_batchs = (len(self.x) + batch_size - 1)// batch_size


    def PadSeq(self, seq):

        if len(seq) > self.max_len:
            return seq[:self.max_len]
        else:
            return seq + [self.pad_token] * (self.max_len - len(seq)) # pad
        
    def __iter__(self):
        perm = np.random.permutation(len(self.x))

        for i in range(0, len(self.x), self.batch_size):
            ids = perm[i : i + self.batch_size]
            batch_x = [self.PadSeq(self.x[j]) for j in ids] # target
            batch_y = [self.PadSeq(self.y[j]) for j in ids] # label

        yield torch.tensor(batch_x), torch.tensor(batch_y) # batch shape [batch_size, max_len]
        
    def __len__(self):
        return self.n_batchs

with open('tokenizers/wiki_token.pkl', 'rb') as f:
    data = pickle.load(f)

print(data[0:5])
print(len(data))

X_train, X_test, y_train, y_test = train_test_split(data[:-1], data[1:], test_size=0.2, random_state=88)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=88)






[[354, 696, 76, 11404, 2821, 318, 257, 1316, 68, 13, 340, 318, 287, 443, 12, 2934, 12, 69, 8132, 287, 262, 3209, 47476, 5011, 287, 5093, 1216, 590, 13], [17006, 570, 261, 4244, 318, 257, 1316, 68, 13, 340, 318, 287, 443, 12, 2934, 12, 69, 8132, 287, 262, 3209, 47476, 5011, 287, 5093, 1216, 590, 13], [354, 559, 487, 454, 12, 7278, 12, 2213, 29658, 318, 257, 1316, 68, 13, 340, 318, 287, 443, 12, 2934, 12, 69, 8132, 287, 262, 3209, 47476, 5011, 287, 5093, 1216, 590, 13], [2395, 457, 391, 4244, 318, 257, 1316, 68, 13, 340, 318, 287, 443, 12, 2934, 12, 69, 8132, 287, 262, 3209, 47476, 5011, 287, 5093, 1216, 590, 13], [49916, 1236, 274, 318, 257, 1316, 68, 13, 340, 318, 287, 443, 12, 2934, 12, 69, 8132, 287, 262, 3209, 47476, 5011, 287, 5093, 1216, 590, 13]]
243883


In [64]:
max_token = max(max(seq) for seq in X_train if len(seq) > 0)
print("Max token ID in X_train:", max_token)

Max token ID in X_train: 50254


In [4]:
# ==== Configuration ==== #
@dataclass
class ModelConfig:
    def __init__(self):
        self.block_size = 8 # context size (min num of tokens in sequence, how long model can see at once)
        self.vocab_size = 10000
        self.max_iters = 100
        self.n_head = 4
        self.n_embd = 32
        self.n_layers = 4
        self.dropout = 0.
        self.bias: bool = True
        self.device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
                                          # switch out when transferring to github or stanage for cuda use

@dataclass
class TrainConfig:
    epochs: int = 150
    batch_size: int = 4
    max_len: int = 100
    pad: int = 0
    grad_accum_steps: int = 1
    max_lr: float = 3e-4
    min_lr: float = 1e-5
    warmup_iters: int = 100
    lr_decay_iters: int = 1000
    save_every: int = 1
    # add optimiser hp


# GPT2 Decoder #

Components:

1. Token Embeddings - Map token ids to dense vecs

2. Positional Encoding - add positional info to embeddings

3. Transformer Blocks (n) - self-attention + feed foward layers

4. Casual masking - prevents peeking into future

5. Final Linear Layer - projects to vocab size (logits)

In [5]:
# ==== GPT2 Decoder ==== #

### UPDATE FOR TORCH USE ###

class GPT2Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.config = config

        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias) # create W attn embds
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)

        # layer norm
        self.ln1 = nn.LayerNorm(config.n_embd)
        self.ln2 = nn.LayerNorm(config.n_embd)

        self.mlp = nn.Sequential(
            nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias), # expand dims (4 * from Attention all u need paper)
            nn.GELU(),
            nn.Linear(4 * config.n_embd, config.n_embd), # project back 
            nn.Dropout(config.dropout)
        )
        
        self.attn_dropout = nn.Dropout(config.dropout)


    def SelfAttention(self,x, mask=None):
        b,t,c = x.shape
        q, k, v = self.c_attn(x).split(self.config.n_embd, dim=2)
        
        q =q.reshape(b, t, self.config.n_head, c//self.config.n_head).permute(0,2,1,3)
        k = k.reshape(b, t, self.config.n_head, c//self.config.n_head).permute(0,2,1,3)
        v = v.reshape(b, t, self.config.n_head, c//self.config.n_head).permute(0,2,1,3)

        score = (q @ k.transpose(-2,-1)) / math.sqrt(k.shape[-1])

        if mask is not None:
            score = score + mask

        a = nn.functional.softmax(score, dim=-1)
        a = self.attn_dropout(a)
        head = a @ v
        out = head.permute(0,2,1,3).reshape(b,t,c) # back to input dims
        out = self.c_proj(out)
        #print("Type of self.SelfAttention(x):", type(out))
        return out
    
    def forward(self, x, mask=None):
        # self attention ad layer norm
        x = x + self.SelfAttention(self.ln1(x), mask)
        x = x + self.mlp(self.ln2(x)) # resudual connections
       
        return x
    

class GPT2Model(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.token_embd = nn.Embedding(config.vocab_size, config.n_embd)
        self.posn_embd = nn.Embedding(config.block_size, config.n_embd)

        self.proj = nn.Linear(config.n_embd, config.vocab_size)

        self.blocks = nn.Sequential(*[GPT2Block(config) for _ in range(config.n_layers)])
        self.fln = nn.LayerNorm(config.n_embd)

    def forward(self, x):
        b, t = x.size()
        tokn_embd = self.token_embd(x)
        posn_idcs = torch.arange(t, device=x.device).unsqueeze(0).expand(b,t)
        posn_embd = self.posn_embd(posn_idcs)

        x = tokn_embd + posn_embd

        x = self.blocks(x)
        x = self.fln(x) # final linear layer
        return self.proj(x) # project back to (enbedding size, vocab size)


class GPTTrainer: # ADD Validation and test 
    def __init__(self, x, y, train_config, model_config, check_points_dir=None):
        self.x, self.y = x, y
        self.train_config = train_config
        self.check_points_dir = check_points_dir

        self.device = model_config.device

        self.data_loader = DataLoader(
                                    self.x, self.y, 
                                    train_config.batch_size, 
                                    train_config.max_len,
                                    train_config.pad
                                    )
        
        # Intitialise model and optimiser
        self.model = GPT2Model(model_config).to(self.device)
        self.optim = torch.optim.AdamW(self.model.parameters(), lr=train_config.max_lr)
        self.criterion = nn.CrossEntropyLoss()


    def step(self, inputs, targets):
        self.model.train()

        inputs = inputs.to(self.device)
        targets = targets.to(self.device)

        logits = self.model(inputs) # foward pass

        b, t, v = logits.shape
        logits = logits.view(b*t, v)
        targets = targets.view(b*t)

        loss = self.criterion(logits, targets)

        # back prop
        self.optim.zero_grad()
        loss.backward()
        self.optim.step()

        return loss.item()

    
    def train(self):
        for epoch in range(self.train_config.epochs):
            total_loss = 0.
            n_batches = 0

            for x, y in self.data_loader:
                #print(f"Batch {n_batches+1}")
                loss_val = self.step(x, y)
                total_loss += loss_val
                n_batches +=1

            avg_loss = total_loss / n_batches
            print(f"Epoch {epoch+1} | Loss: {avg_loss:.3f}")

            # save checpoints
            if self.check_points_dir and (epoch + 1) % self.train_config.save_every ==0:
                torch.save(f"{self.check_points_dir}/model_epoch_{epoch+1}.npz", **self.model.state_dict())

 
class GenerateGPT:
    def __init__(self, model, tokeniser, prompt, max_len=50):
        super.__init__()
 


# Text Generation # 

Predicts next token from the given context. 

1. Input [cat sat on mat] -> token representation 

2. Feed tokens to the trained model. It will output tensor with shape [batch_size, seq_len, vocab_size], which will have the probability distribution over the vocab at each position. Only care about the last position to predict next token.

3. Pick the token. From the final distribution: Argmax for highest prob, or sample from the distribution (more creative) or use top-k/top-p (nucleus) sampling to balance diversity and equity. Then simply append token to sequence.

4. Repeat, feed the model the updated sequence back in and repeat 2 and 3 until a max length is reached or model generates and end of sequence token.

### Casual Masking ###
In training and generation, model cant peak into the future.

Tests:
- Different prompt lengths
- Adjust temperature (controls randomness in sampling)
- Add stop conditions (e.g if it generates a period or newline)
- see how model behaves w/ different types of prompt

In [63]:
print(X_train[0])

[547, 7323, 352, 1582, 2363, 220, 1973, 13, 780, 286, 262, 14903, 286, 32558, 12858, 11, 262, 45508, 4712, 26843, 5443, 355, 340, 14707, 13, 355, 262, 2587, 1626, 262, 45508, 4712, 38784, 11, 262, 23235, 1626, 340, 2540, 284, 46592, 351, 3649, 8373, 11, 23202, 511, 37892, 2568, 656, 4894, 13, 262, 3641, 11, 810, 749, 286, 262, 2347, 7723, 11, 2627, 6481, 37546, 621, 262, 7346, 1221, 13, 625, 546, 1802, 11, 830, 812, 11, 257, 3024, 11, 15715, 1237, 455, 283, 2540, 379, 262, 7372, 13, 262, 13325, 287, 11539, 1043, 287, 19999, 2737, 743, 307, 262]


In [ ]:

#loader = DataLoader(x, y, batch_size=8, max_len=100, pad_token=0)
model_config = ModelConfig()
train_config = TrainConfig()

trainGPT = GPTTrainer(X_train, y_train, train_config, model_config, check_points_dir=None)
trainGPT.train()

Epoch 1 | Loss: 8.862
Epoch 2 | Loss: 8.315
Epoch 3 | Loss: 8.418
Epoch 4 | Loss: 8.282
Epoch 5 | Loss: 8.473
Epoch 6 | Loss: 8.195
Epoch 7 | Loss: 8.418
Epoch 8 | Loss: 8.575
Epoch 9 | Loss: 8.499
Epoch 10 | Loss: 8.290
Epoch 11 | Loss: 8.601
Epoch 12 | Loss: 8.122
Epoch 13 | Loss: 8.022
Epoch 14 | Loss: 8.106
Epoch 15 | Loss: 8.061
Epoch 16 | Loss: 8.068
Epoch 17 | Loss: 7.913
Epoch 18 | Loss: 8.426
Epoch 19 | Loss: 8.169
Epoch 20 | Loss: 7.748
Epoch 21 | Loss: 8.316
Epoch 22 | Loss: 8.177
Epoch 23 | Loss: 7.948
Epoch 24 | Loss: 7.724
Epoch 25 | Loss: 7.744
Epoch 26 | Loss: 8.272
Epoch 27 | Loss: 7.520
Epoch 28 | Loss: 7.863
Epoch 29 | Loss: 7.880
Epoch 30 | Loss: 8.198
Epoch 31 | Loss: 7.550
Epoch 32 | Loss: 7.971
Epoch 33 | Loss: 8.413
Epoch 34 | Loss: 8.177
Epoch 35 | Loss: 7.891
Epoch 36 | Loss: 7.588
Epoch 37 | Loss: 7.532
Epoch 38 | Loss: 7.806
Epoch 39 | Loss: 7.553
Epoch 40 | Loss: 7.793
Epoch 41 | Loss: 8.050
Epoch 42 | Loss: 7.663
Epoch 43 | Loss: 7.397
Epoch 44 | Loss: 7.9

In [ ]:
# ==== VAE configs ==== #


    

In [156]:
# ==== VAE w/ BiLSTM ==== #

class VAEBiLSTM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config

        self.embd = nn.Embedding(config.vocab_size, config.embd_dim, config.pad_idx)
        #assert config.input_dim == config.embd_dim 

        # biLSTM encoder
        self.encoder = nn.LSTM(
            config.embd_dim, 
            config.hidden_dim, 
            config.n_layers, 
            batch_first=True, 
            bidirectional=True) # returns tensor (batch size, seq len, 2 * hidden size)
        
        self.h_to_mu = nn.Linear(2 * config.hidden_dim, config.latent_dim)
        self.h_to_logVar = nn.Linear(2 * config.hidden_dim, config.latent_dim)

        # Decoder (transformer)
        self.z_embd = nn.Linear(config.latent_dim, config.embd_dim)

        self.decode = nn.TransformerDecoder(
            nn.TransformerDecoderLayer(d_model=config.embd_dim, 
                                       nhead=config.decoder_heads,
                                       dropout=config.dropout),
                                       num_layers=config.n_layers
                                       )
         # maybe if time try RoPE posn embeddings

        # positional embd for target
        self.posn_embd_target = nn.Embedding(config.block_size, config.embd_dim) #n_embd or embd_dim?

        self.posn_embd = nn.Embedding(config.block_size, config.embd_dim)
        self.token_embd = nn.Embedding(config.vocab_size, config.embd_dim)
        self.proj_out = nn.Linear(config.embd_dim, config.vocab_size)

    def Encode(self, x):
        x = self.embd(x) # [batch, seq len, embd dim]
        #print(x.shape)

        out, (hn, cn) = self.encoder(x)

        ht = torch.concat((hn[-2], hn[-1]), dim=1) # summary rep, last back and forward layers

        mu = self.h_to_mu(ht)
        logVar = self.h_to_logVar(ht)

        return mu, logVar
    
    def Reparameterise(self, mu, logVar):
        std = torch.exp(0.5 * logVar)
        eps = torch.randn_like(std)
        z = mu + eps*std
        return z # z shape [batch, hidden]
    
    def Decode(self, z, target_seq):
        proj_z = self.z_embd(z) # [batch (input shape), embd]

        # embed target seq
        target_embd = self.embd(target_seq) #[batch, seq, embd]
        target_embd = target_embd.transpose(0,1) #[seq, batch, embd] 

  
        b, t = target_seq.size()
       
        posn_idc = torch.arange(t, device=z.device).unsqueeze(0).expand(b,t)
        posn_embd = self.posn_embd_target(posn_idc)
        posn_embd = posn_embd.transpose(0,1)
   
        #print("posn",posn_embd.size())
        #print('target',target_embd.size())
        target = target_embd + posn_embd
        target = target.transpose(0,1) # transformer needs [seq, batch, embd]

        proj_z = proj_z.unsqueeze(1).expand(-1, t, -1) # [batch, t, embd_dim]
        #print('proj_z size', proj_z.size())
        memory =  proj_z # projected z for conditioning [t, batch, embd]
        #print('memory size', memory.size())
        #print('target size', target.size())
      
        out = self.decode(tgt=target, memory=memory) # [seq, batch, embd]

        # project to output shape [seq, batch, embd] -> [seq, batch, vocab len]
        out = self.proj_out(out)

        return out

    def forward(self, x, target_seq):
        mu, logvar = self.Encode(x)
        z = self.Reparameterise(mu, logvar)
        return z, self.Decode(z, target_seq), mu, logvar

    
        


In [160]:
class TrainBiLSTMVAE(nn.Module):
    def __init__(self, x, y, train_config, model_config):
        super().__init__()
        self.x = x
        self.y = y

        self.train_config = train_config
        self.model_config = model_config

        self.device = model_config.device

        self.loader = DataLoader(
            self.x, self.y, 
            model_config.batch_size,
            train_config.max_len,
            train_config.pad_token
        ) 

        # int model
        self.model = VAEBiLSTM(model_config).to(self.device)
        self.optim = torch.optim.AdamW(self.model.parameters(), lr=train_config.max_lr)

        self.global_step = 0
        
    def LossFunc(self, recon_x, x, mu, logvar):
        total_steps = self.train_config.epochs * len(self.loader)

        beta = min(1.0, self.global_step / (total_steps * self.train_config.kl_speed))
        #beta = 0


        recon_x = recon_x.view(-1, self.model_config.vocab_size)
        x = x.view(-1)

        CE = F.cross_entropy(recon_x, x, ignore_index=self.model_config.pad_idx, reduction='mean')

        KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
        KLD = KLD / recon_x.size(0)  # normalize over batch
        print("KL:", KLD.item(), "CE:", CE.item())


        return CE + beta * KLD


    def TrainStep(self):
        self.model.train()
        self.device = torch.device("cpu") # need to change this for testing on stanage for cude
        self.model.to("cpu") #mps compatibility issues with lstm module on torch
        epoch_losses = []

        for epoch in range(self.train_config.epochs):
            running_loss = 0.0

            for batch_x, _ in self.loader:
                self.global_step += 1
                batch_x = batch_x.long().to(self.device)
                #print("First batch shape:", batch_x.shape)


                self.optim.zero_grad()

                z, recon_x, mu, logvar = self.model(batch_x, batch_x)  # input = target for teacher forcing
                loss = self.LossFunc(recon_x, batch_x, mu, logvar)

                loss.backward()
                self.optim.step()

                running_loss += loss.item()

            avg_loss = running_loss / len(self.loader)
            epoch_losses.append(avg_loss)
            print(f"Epoch {epoch+1}/{self.train_config.epochs} | Loss: {avg_loss:.4f}")

        return epoch_losses 




In [161]:
@dataclass
class VAEConfig:
    batch_size: int = 8
    hidden_dim: int = 1024
    embd_dim: int = 128
    latent_dim:int = 64
    input_dim: int = 64
    n_layers: int = 1
    pad_idx: int = 0
    vocab_size: int = 50255 # max token val
    decoder_heads: int = 8
    dropout: float = 0.1 # dropout for transformer decoder
    block_size: int = 100 # match seqence length
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
                        # switch out when transferring to github or stanage for cuda use


@dataclass
class VAETrainConfig:
    epochs: int = 30
    max_lr: float = 5e-4
    min_lr: float = 1e-5
    device:  torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    max_len: int = 100
    pad_token: int = 0
    kl_speed = 1
   # batch_size:int = 8

In [162]:
print(len(X_train))

vae_trainer = TrainBiLSTMVAE(X_train, y_train, VAETrainConfig, VAEConfig)
losses = vae_trainer.TrainStep()

146328
KL: 0.0005120969726704061 CE: 11.01603889465332
Epoch 1/30 | Loss: 0.0006
KL: 0.0006616286700591445 CE: 10.896020889282227
Epoch 2/30 | Loss: 0.0006
KL: 0.0008665387285873294 CE: 10.686738967895508
Epoch 3/30 | Loss: 0.0006
KL: 0.0015674046007916331 CE: 10.54519271850586
Epoch 4/30 | Loss: 0.0006
KL: 0.0020198815036565065 CE: 10.409372329711914
Epoch 5/30 | Loss: 0.0006
KL: 0.00527367414906621 CE: 10.295669555664062
Epoch 6/30 | Loss: 0.0006
KL: 0.008542085997760296 CE: 10.18830394744873
Epoch 7/30 | Loss: 0.0006
KL: 0.01544257439672947 CE: 10.18809700012207
Epoch 8/30 | Loss: 0.0006
KL: 0.11881127208471298 CE: 10.14050579071045
Epoch 9/30 | Loss: 0.0006
KL: 0.035955000668764114 CE: 9.931495666503906
Epoch 10/30 | Loss: 0.0005
KL: 0.033493366092443466 CE: 9.690995216369629
Epoch 11/30 | Loss: 0.0005
KL: 0.03108862228691578 CE: 9.680652618408203
Epoch 12/30 | Loss: 0.0005
KL: 0.020024003461003304 CE: 9.57498550415039
Epoch 13/30 | Loss: 0.0005
KL: 0.01652737893164158 CE: 9.555003

In [ ]:
from sklearn.decomposition import PCA


Visualise latents. look for semantic similarity, helps to show if simpler inputs are mapped to more compact and confident region are formed because of irreducile and entropy. 

Detects posterior collapse (all z are crammed into small region)

Training progression tracked, can see how it evolves across epochs

Can plot entropy or difficulty vs position in latent space

Work out how to color code for semantic similarity, mark sequences by curriculum, possibly show evolution over epochs

# VAE first then curriculum or curriculum then VAE ? #

reduce timie to convergence

potentially reduce overfitting 

potentially outperform naivly trained models 